# Paper 1: Regime Labels Are Not Representation-Invariant

This notebook demonstrates the core finding of **Paper 1** (Zheng, Low & Wang, 2026):
regime labels produced by clustering models (e.g., GMM) are **not stable** across
different feature representations of the same underlying price series.

We test this by:
1. Building three distinct factor sets from the same synthetic price data
2. Fitting a 3-state GMM on each
3. Comparing the resulting labels pairwise via **ARI** (partition stability) and **Spearman** (ordering consistency)
4. Applying the two-layer verdict from the paper

No real market data or IB connection is needed — we generate a synthetic price series with regime-like behavior.

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

from mrv.data.factors import build_factors, log_returns, volatility, register_factor
from mrv.data.normalize import normalize
from mrv.models import fit, register_model
from mrv.validator.metrics import (
    ari, ami, variation_of_information, ordering_consistency,
    ARI_THRESHOLD, SPEARMAN_THRESHOLD,
)
from mrv.validator.rep import RepValidator
from mrv.pipeline import run, validate, report

print(f"ARI threshold:      {ARI_THRESHOLD}")
print(f"Spearman threshold: {SPEARMAN_THRESHOLD}")

## 2. Synthetic Data (no IB needed)

We generate a 1000-observation synthetic price series with two regimes:
- **Calm** (60% of observations): low volatility, slight upward drift
- **Stressed** (40% of observations): high volatility, negative drift

This gives us a realistic-looking series to test whether regime labels are stable
across different feature representations.

In [ ]:
np.random.seed(42)

n_obs = 1000
dates = pd.bdate_range("2020-01-02", periods=n_obs, freq="B")

# Regime assignments: 60% calm, 40% stressed (in blocks)
regime = np.zeros(n_obs, dtype=int)
block_starts = [0, 200, 400, 550, 700, 850]
block_regimes = [0, 1, 0, 1, 0, 1]  # alternating calm/stressed
for i, (start, r) in enumerate(zip(block_starts, block_regimes)):
    end = block_starts[i + 1] if i + 1 < len(block_starts) else n_obs
    regime[start:end] = r

# Generate returns conditional on regime
mu = np.where(regime == 0, 0.0003, -0.0005)   # drift
sigma = np.where(regime == 0, 0.008, 0.025)    # volatility
returns = np.random.normal(mu, sigma)

# Convert to price level (start at 100)
price = 100.0 * np.exp(np.cumsum(returns))
price_series = pd.Series(price, index=dates, name="SyntheticAsset")

# Also create an OHLCV DataFrame (needed by RepValidator)
price_df = pd.DataFrame({
    "Open": price_series * (1 + np.random.normal(0, 0.001, n_obs)),
    "High": price_series * (1 + np.abs(np.random.normal(0, 0.005, n_obs))),
    "Low":  price_series * (1 - np.abs(np.random.normal(0, 0.005, n_obs))),
    "Close": price_series,
    "Volume": np.random.randint(1_000_000, 10_000_000, n_obs),
}, index=dates)

print(f"Observations: {n_obs}")
print(f"Calm:    {(regime == 0).sum()} ({(regime == 0).mean():.0%})")
print(f"Stressed: {(regime == 1).sum()} ({(regime == 1).mean():.0%})")

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(dates, price, color="steelblue", linewidth=0.8)
ax1.set_ylabel("Price")
ax1.set_title("Synthetic Price Series with Regime Structure")

colors = np.where(regime == 0, "#2ecc71", "#e74c3c")
ax2.bar(dates, regime, color=colors, width=1.5)
ax2.set_ylabel("Regime")
ax2.set_yticks([0, 1])
ax2.set_yticklabels(["Calm", "Stressed"])

fig.tight_layout()
plt.show()

## 3. Define Three Factor Sets

Paper 1 tests invariance across multiple feature representations. We define three sets:

| Set | Factors | Rationale |
|-----|---------|----------|
| **A** | vol, drawdown, maxdd, var, cvar | Full tail-risk suite (default) |
| **B** | vol, drawdown, var, cvar | Drop max drawdown (minor reduction) |
| **C** | real_skew, vol_stab, var, cvar | Higher-order moments replace drawdown |

We build factors from the same price series and normalize with `rolling_zscore`.

In [ ]:
# Factor set definitions (using aliases)
set_a_factors = ["vol", "drawdown", "maxdd", "var", "cvar"]
set_b_factors = ["vol", "drawdown", "var", "cvar"]
set_c_factors = ["real_skew", "vol_stab", "var", "cvar"]

# Window configuration
windows = {
    "vol_window": 20,
    "drawdown_window": 60,
    "tail_window": 60,
    "tail_alpha": 0.05,
    "skew_window": 60,
    "stability_window": 60,
}

# Build raw factors
raw_a = build_factors(price_series, factors=set_a_factors, windows=windows)
raw_b = build_factors(price_series, factors=set_b_factors, windows=windows)
raw_c = build_factors(price_series, factors=set_c_factors, windows=windows)

# Normalize with rolling z-score
norm_a = normalize(raw_a, mode="rolling_zscore", window=120).dropna()
norm_b = normalize(raw_b, mode="rolling_zscore", window=120).dropna()
norm_c = normalize(raw_c, mode="rolling_zscore", window=120).dropna()

print("Set A columns:", list(norm_a.columns))
print("Set B columns:", list(norm_b.columns))
print("Set C columns:", list(norm_c.columns))
print(f"\nUsable observations after normalization: A={len(norm_a)}, B={len(norm_b)}, C={len(norm_c)}")
print("\nSet A sample:")
norm_a.head()

## 4. Fit Regime Model (K=3 GMM)

We fit a 3-state Gaussian Mixture Model on each normalized factor set.
The `fit()` function returns hard labels (integer state assignments).

In [ ]:
labels_a = fit(norm_a, model="gmm", n_states=3)
labels_b = fit(norm_b, model="gmm", n_states=3)
labels_c = fit(norm_c, model="gmm", n_states=3)

all_labels = {
    "Set A (full tail-risk)": labels_a,
    "Set B (no maxdd)": labels_b,
    "Set C (higher-order)": labels_c,
}

# Print state distributions
for name, lbl in all_labels.items():
    unique, counts = np.unique(lbl, return_counts=True)
    dist = ", ".join(f"state {s}: {c} ({c/len(lbl):.1%})" for s, c in zip(unique, counts))
    print(f"{name}: {dist}")

## 5. Pairwise ARI Matrix

The **Adjusted Rand Index (ARI)** measures partition agreement between two clusterings.
- ARI = 1.0: perfect agreement
- ARI ~ 0.0: random agreement
- ARI < 0: worse than random

Paper 1 uses a threshold of **ARI >= 0.65** (Steinley, 2004) for acceptable partition recovery.

In [ ]:
names = list(all_labels.keys())
n = len(names)
ari_matrix = pd.DataFrame(np.eye(n), index=names, columns=names)

for (na, la), (nb, lb) in combinations(all_labels.items(), 2):
    val = ari(la, lb)
    ari_matrix.loc[na, nb] = val
    ari_matrix.loc[nb, na] = val

print("ARI Pairwise Matrix:")
print(ari_matrix.round(3).to_string())

offdiag = ari_matrix.values[np.triu_indices(n, k=1)]
print(f"\nMean ARI: {np.mean(offdiag):.3f}")
print(f"Min  ARI: {np.min(offdiag):.3f}")
print(f"Threshold: {ARI_THRESHOLD}")
print(f"Partition stable? {'YES' if np.mean(offdiag) >= ARI_THRESHOLD else 'NO'}")

## 6. Ordering Consistency (Spearman)

Even when partitions differ, the **risk ordering** may be preserved.
Each state is ranked by its mean risk (using realized volatility as proxy).
Observations are then mapped to their state's risk rank, and we compute
the **Spearman correlation** of these rank sequences.

Threshold: **Spearman >= 0.85** indicates stable risk ordering.

In [ ]:
# Use realized volatility as risk proxy
ret = log_returns(price_series)
vol_proxy = volatility(ret, window=20, annualize=False).dropna().values

sp_matrix = pd.DataFrame(np.eye(n), index=names, columns=names)

for (na, la), (nb, lb) in combinations(all_labels.items(), 2):
    nc = min(len(la), len(lb), len(vol_proxy))
    val = ordering_consistency(la[:nc], lb[:nc], vol_proxy[:nc])
    sp_matrix.loc[na, nb] = val
    sp_matrix.loc[nb, na] = val

print("Ordering Consistency (Spearman) Matrix:")
print(sp_matrix.round(3).to_string())

sp_offdiag = sp_matrix.values[np.triu_indices(n, k=1)]
print(f"\nMean Spearman: {np.mean(sp_offdiag):.3f}")
print(f"Min  Spearman: {np.min(sp_offdiag):.3f}")
print(f"Threshold: {SPEARMAN_THRESHOLD}")
print(f"Ordering stable? {'YES' if np.mean(sp_offdiag) >= SPEARMAN_THRESHOLD else 'NO'}")

## 7. Two-Layer Verdict

Paper 1 defines a **two-layer test**:

1. **Partition stability**: Mean ARI >= 0.65 (exact cluster boundaries are similar)
2. **Ordering stability**: Mean Spearman >= 0.85 (risk ranking is preserved)

A representation is considered **invariant** only if both layers pass.
In practice, ordering stability often holds even when partition stability fails,
suggesting that the risk *ranking* is more robust than exact cluster *boundaries*.

In [ ]:
mean_ari = float(np.mean(offdiag))
mean_sp = float(np.mean(sp_offdiag))

partition_pass = mean_ari >= ARI_THRESHOLD
ordering_pass = mean_sp >= SPEARMAN_THRESHOLD

print("=" * 55)
print("  TWO-LAYER REPRESENTATION INVARIANCE VERDICT")
print("=" * 55)
print(f"  Layer 1 - Partition stability (ARI):")
print(f"    Mean ARI = {mean_ari:.3f}  (threshold = {ARI_THRESHOLD})")
print(f"    Result:  {'PASS' if partition_pass else 'FAIL'}")
print()
print(f"  Layer 2 - Ordering stability (Spearman):")
print(f"    Mean Spearman = {mean_sp:.3f}  (threshold = {SPEARMAN_THRESHOLD})")
print(f"    Result:  {'PASS' if ordering_pass else 'FAIL'}")
print()
print(f"  Overall: {'INVARIANT' if (partition_pass and ordering_pass) else 'NOT INVARIANT'}")
print("=" * 55)

if not partition_pass and ordering_pass:
    print("\nNote: Ordering is stable but partitions differ.")
    print("This is the typical Paper 1 finding: risk ranking is more")
    print("robust than exact cluster boundaries across representations.")

## 8. ARI Heatmap Visualization

Color-coded heatmap of the pairwise ARI matrix. Cells below the threshold
appear in red/yellow, indicating representation sensitivity.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
data = ari_matrix.values.astype(float)
im = ax.imshow(data, vmin=-0.1, vmax=1.0, cmap="RdYlGn", aspect="auto")

short_labels = ["Set A", "Set B", "Set C"]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_yticklabels(short_labels, fontsize=10)

# Annotate cells
for i in range(n):
    for j in range(n):
        v = data[i, j]
        color = "white" if v < 0.4 else "black"
        weight = "bold" if i != j else "normal"
        ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                fontsize=13, fontweight=weight, color=color)

ax.set_title(f"Cross-Representation ARI\n(threshold = {ARI_THRESHOLD})",
             fontsize=13, fontweight="bold", pad=12)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("ARI", fontsize=11)
cbar.ax.axhline(y=ARI_THRESHOLD, color="black", linewidth=1.5, linestyle="--")

fig.tight_layout()
plt.show()

## 9. Ablation: Standardization Effect

A key finding of Paper 1 is that **normalization choice** itself affects regime labels.
Here we compare the same factor set (Set A) with and without `rolling_zscore`
normalization. Even this single preprocessing change can shift cluster boundaries.

In [ ]:
# Set A: normalized vs raw (no normalization)
raw_a_unscaled = raw_a.dropna()

labels_a_scaled = fit(norm_a, model="gmm", n_states=3)
labels_a_unscaled = fit(raw_a_unscaled, model="gmm", n_states=3)

# Align lengths
nc = min(len(labels_a_scaled), len(labels_a_unscaled))
ari_standardization = ari(labels_a_scaled[:nc], labels_a_unscaled[:nc])

vol_nc = vol_proxy[:nc]
sp_standardization = ordering_consistency(
    labels_a_scaled[:nc], labels_a_unscaled[:nc], vol_nc
)

print("Ablation: Standardization Effect (Set A)")
print("=" * 45)
print(f"  rolling_zscore vs raw (no normalization)")
print(f"  ARI:      {ari_standardization:.3f}  (threshold = {ARI_THRESHOLD})")
print(f"  Spearman: {sp_standardization:.3f}  (threshold = {SPEARMAN_THRESHOLD})")
print()

if ari_standardization < ARI_THRESHOLD:
    print("  Conclusion: Normalization alone changes the partition.")
    print("  This confirms Paper 1's finding that regime labels are")
    print("  sensitive to preprocessing, not just factor selection.")
else:
    print("  Conclusion: Normalization did not significantly alter")
    print("  the partition in this synthetic example.")

## 10. Using RepValidator (Config-Driven)

For production use, `RepValidator` handles the full pipeline:
build factors, normalize, fit, compare, save results and heatmap.

You can pass a config dict directly (no YAML file needed).

In [ ]:
config = {
    "factors": {
        "vol_window": 20,
        "drawdown_window": 60,
        "tail_window": 60,
        "tail_alpha": 0.05,
        "skew_window": 60,
        "stability_window": 60,
    },
    "normalize": {
        "mode": "rolling_zscore",
        "window": 120,
    },
    "validator": {
        "report_dir": "reports",
        "rep": {
            "model": "gmm",
            "n_states": 3,
            "factors": [
                ["vol", "drawdown", "maxdd", "var", "cvar"],
                ["vol", "drawdown", "var", "cvar"],
                ["real_skew", "vol_stab", "var", "cvar"],
            ],
        },
    },
}

# Pass the synthetic Close price series (RepValidator expects {asset: Series})
result = RepValidator(config).validate(
    prices={"SyntheticAsset": price_df["Close"]}
)

print("RepValidator output keys:", list(result.keys()))
print(f"Run dir: {result['run_dir']}")

# Display the validator's ARI matrix
asset_result = result["assets"]["SyntheticAsset"]
print(f"\nMean ARI:      {asset_result['mean_ari']:.3f}")
print(f"Min  ARI:      {asset_result['min_ari']:.3f}")
print(f"Mean Spearman: {asset_result['mean_spearman']:.3f}")
print(f"\nARI matrix:")
print(asset_result["ari_matrix"].round(3).to_string())

## 11. Config-File Workflow

For production, define the test in a YAML config file and run via CLI.

**config.yaml** (relevant section):
```yaml
factors:
  vol_window: 20
  drawdown_window: 60
  tail_window: 60
  tail_alpha: 0.05

normalize:
  mode: rolling_zscore
  window: 120

validator:
  report_dir: reports
  rep:
    assets:
      SPY: data/SPY_5m.csv
      CL:  data/CL_5m.csv
    model: gmm
    factors:
      - [vol, drawdown, maxdd, var, cvar]
      - [vol, drawdown, var, cvar]
      - [real_skew, vol_stab, var, cvar]
```

**CLI command:**
```bash
python run.py run config.yaml rep
```

This will:
1. Load price data from the specified CSV paths
2. Build factors for each of the 3 factor sets
3. Normalize with rolling z-score
4. Fit 3-state GMM on each
5. Compute ARI + Spearman matrices
6. Save JSON results and ARI heatmap PNGs
7. Generate a PDF report

In [ ]:
# The equivalent Python call for the CLI command above:
# from mrv.pipeline import run
# pdf_path = run("config.yaml", validator="rep")

print("To run with real market data:")
print("  python run.py run config.yaml rep")
print()
print("This notebook demonstrated the core Paper 1 workflow:")
print("  1. Same price series, three different factor representations")
print("  2. GMM clustering on each -> pairwise ARI + Spearman")
print("  3. Two-layer verdict: partition stability + ordering stability")
print("  4. Ablation showing standardization sensitivity")
print("  5. Config-driven RepValidator for production use")